RETRIEVER

In [1]:
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader,get_response_synthesizer
from llama_index.core import SimpleDirectoryReader
from llama_index.core import Settings

Settings.chunk_size = 128
Settings.chunk_overlap = 50

documents = SimpleDirectoryReader("/home/jjh_test/llama_index/data").load_data()
index = VectorStoreIndex.from_documents(documents)
retriever = index.as_retriever(verbose=True, similarity_top_k=2)
response_synthesizer = get_response_synthesizer(
    response_mode="compact",
)
query_engine = index.as_query_engine()

In [2]:
eval_q_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-q").load_data()

In [3]:
q_data = eval_q_data[0].text

questions = [line.strip().strip('"') for line in q_data.split('\n') if line.strip()]

In [4]:
retriever_result = {}
search_results = []

for count, question in enumerate(questions, start=1):
    search_this = f"{question}"
    search_result = retriever.retrieve(search_this)
    search_results = []
    
    for i in range(retriever.similarity_top_k):
        search_results.append(search_result[i].get_text())

    retriever_result[count] = search_results

EMBEDDING MODEL

In [5]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

GENERATOR

In [6]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex.from_documents(documents)

In [7]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import sys

torch.random.manual_seed(0)
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    torch_dtype="auto", 
    trust_remote_code=True, max_length=500
)
assert torch.cuda.is_available(), "This model needs a GPU to run ..."
device = torch.cuda.current_device()
model = model.to(device)
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
responses=[]

In [9]:
for count, question in enumerate(questions, start=1):
    retriever_context_list = retriever_result.get(count, [])
    retriever_context = ' '.join(retriever_context_list)
    
    query = f" Given the context: '{retriever_context}', determine if the following statement is 'True' or 'False': {question}. Start your response with 'True' or 'False'. Answer:"
    
    model_inputs = tokenizer(query, return_tensors="pt").to("cuda")
    generated_ids = model.generate(**model_inputs)
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
    aa = response.find("Answer:")
    if aa != -1:
        response = response[aa + 7:aa + 14]  # Extract only 6 characters after "Answer: "
    else:
        response = response[:6]  # Fallback in case "Answer:" is not found

    responses.append(response)

    print(count, response)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


1  False.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


2  False.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

72  True. 


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


73  False.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


74  True. 
75  True. 


EVALUATOR

In [10]:
eval_a_data = SimpleDirectoryReader("/home/jjh_test/llama_index/eval-a").load_data()

In [11]:
a_data = eval_a_data[0].text

answers = [line.strip().strip('"') for line in a_data.split('\n') if line.strip()]

In [12]:
responses_str = []

In [13]:
for response in responses:
    response_str = str(response)
    if "True" in response_str:
        response_str = "True"
    elif "False" in response_str:
        response_str = "False"
    
    responses_str.append(response_str)

In [14]:
correct_count=0
number=0

for answer, response, response_str, question in zip(answers, responses, responses_str, questions,):
    number+=1

    if answer == response_str:
        correct_count += 1
    else: 
        print("<<wrong>>\n", number, question)
        print("RESPONSE", response)
        print("CORRECT ANSWER", answer)
        print()
print(f"correct_count: {correct_count}")

<<wrong>>
 1 True or False: The discovery of a fourth spatial dimension was made through advancements in theoretical physics.
RESPONSE  False.
CORRECT ANSWER True

<<wrong>>
 6 True or False: The discovery of a fourth spatial dimension led to the concept of a liquid atmosphere replacing Earth's air.
RESPONSE  False.
CORRECT ANSWER True

<<wrong>>
 10 True or False: The discovery of exotic matter with negative mass is a direct result of fourth-dimensional research.
RESPONSE  False.
CORRECT ANSWER True

<<wrong>>
 23 True or False: Human genetic modifications are necessary for extracting oxygen from a liquid atmosphere due to fourth-dimensional advancements.
RESPONSE  False.
CORRECT ANSWER True

<<wrong>>
 64 True or False: Advanced nanotechnology is essential for maintaining the proper oxygen levels in Earth's liquid atmosphere.
RESPONSE  False.
CORRECT ANSWER True

<<wrong>>
 70 True or False: Researchers have successfully altered atomic bonds to convert gases into a breathable liquid.

In [15]:
accuracy = (correct_count / len(questions)) * 100
print(f"Total Questions: {len(questions)}")
print(f"Correct Answers: {correct_count}")
print(f"Accuracy: {accuracy:.2f}%")

Total Questions: 75
Correct Answers: 66
Accuracy: 88.00%
